In [1]:
import sqlite3, json
import pandas as pd

In [3]:
conn = sqlite3.connect('d:/LEE/AI_team/github/Vision_AI_RnD_team/projects/test_project/ais_weather/korea_weather.db', check_same_thread=False)
weather_cursor = conn.cursor()

query = """
        SELECT 
            b.지점명,
            b.latitude, 
            b.longitude, 
            w.*
        FROM 
            weather_buoy AS w
        JOIN 
            buoy_position AS b ON w.지점 = b.지점
        WHERE 
            w.일시 LIKE ?
    """

# 3. 쿼리 실행
current_hour_str = '2025-12-10 3'
search_param = f"{current_hour_str}:%"
weather_cursor.execute(query, (search_param,))
rows = weather_cursor.fetchall()

col_names = [desc[0] for desc in weather_cursor.description]
df = pd.DataFrame(rows, columns=col_names)

# 3. 컬럼 슬라이싱 (Pandas에서 처리)
target_cols = [0, 1, 2] + list(range(5, len(df.columns)))
final_df = df.iloc[:, target_cols]

target_columns = [
    "풍속(m/s)", "풍향(deg)", "GUST풍속(m/s)", "현지기압(hPa)", 
    "습도(%)", "기온(°C)", "수온(°C)", "최대파고(m)", 
    "유의파고(m)", "평균파고(m)", "파주기(sec)", "파향(deg)"
]

weather_avg = final_df[target_columns].mean().round(2)

weather_data = json.dumps(weather_avg.to_dict(), ensure_ascii=False, indent=4, default=str)

In [4]:
print(weather_data)

{
    "풍속(m/s)": 4.97,
    "풍향(deg)": 164.32,
    "GUST풍속(m/s)": 6.47,
    "현지기압(hPa)": 1026.65,
    "습도(%)": 59.35,
    "기온(°C)": 9.63,
    "수온(°C)": 14.84,
    "최대파고(m)": 0.9,
    "유의파고(m)": 0.57,
    "평균파고(m)": 0.39,
    "파주기(sec)": 5.51,
    "파향(deg)": 173.54
}
